In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.append(str(Path("..").resolve()))
import importlib, src.features
importlib.reload(src.features)
from src.features import load_m3, TOKEN
 
df, docs, y, groups = load_m3("../data/raw/evtx_data.csv")

In [4]:
# Thí nghiệm chính: ngẫu nhiên và theo nhóm

# Viết make_model(C=1.0) trả Pipeline TF-IDF (TOKEN, min_df=2) + LR (C, class_weight="balanced", max_iter=3000).
# Tạo random_cv (StratifiedKFold) và group_cv (StratifiedGroupKFold), cùng 5 fold, xáo trộn, random_state=42.
# Chạy cross_validate với hai chỉ số f1_macro và accuracy, in trung bình ± độ lệch.
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate
 
def make_model(C=1.0):
    return Pipeline([
        ("tfidf", TfidfVectorizer(token_pattern=TOKEN, min_df=2)),
        ("clf", LogisticRegression(C=C, class_weight="balanced", max_iter=3000))])
 
random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
group_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# •cross_validate(..., scoring=["f1_macro", "accuracy"]) trả dict có khóa test_f1_macro, test_accuracy.
r = cross_validate(make_model(), docs, y, cv=random_cv,
                   scoring=["f1_macro", "accuracy"])
g = cross_validate(make_model(), docs, y, cv=group_cv, groups=groups,
                   scoring=["f1_macro", "accuracy"])
for name, res in [("ngẫu nhiên", r), ("theo nhóm ", g)]:
    f1, acc = res["test_f1_macro"], res["test_accuracy"]
    print(f"{name}: macro-F1 {f1.mean():.3f} +/- {f1.std():.3f}"
          f" | accuracy {acc.mean():.3f}")



ngẫu nhiên: macro-F1 0.803 +/- 0.009 | accuracy 0.841
theo nhóm : macro-F1 0.206 +/- 0.033 | accuracy 0.289


In [5]:
# Baseline và đếm file "lộ đề"
# Tính macro-F1 của Dummy với group_cv. Rồi với vòng đầu tiên của random_cv,
# đếm bao nhiêu file ở phần chấm cũng có event ở phần train.
from sklearn.dummy import DummyClassifier
 
d = cross_validate(DummyClassifier(strategy="most_frequent"), docs, y,
                   cv=group_cv, groups=groups, scoring="f1_macro")
print("baseline:", round(d["test_score"].mean(), 3))
 
for i, (tr, te) in enumerate(random_cv.split(docs, y), 1):
    # file có mặt ở CẢ 2 phía
    shared = set(groups.iloc[te]) & set(groups.iloc[tr]) # Các phần tử có ở cả a và b.
    n_test = groups.iloc[te].nunique()
    print(f"ngẫu nhiên, fold {i}: {len(shared)}/{n_test} file test cũng có ở train")
    break

baseline: 0.027
ngẫu nhiên, fold 1: 146/151 file test cũng có ở train


In [6]:
#  GroupKFold và StratifiedGroupKFold

# Viết c2_per_fold(cv) đếm số file Command and Control trong phần chấm của từng fold. So hai bộ chia.
from sklearn.model_selection import GroupKFold
 
def c2_per_fold(cv):
    out = []
    for train_idx, test_idx in cv.split(docs, y, groups):
        test_files = df.iloc[test_idx].groupby("EVTX_FileName")["EVTX_Tactic"].first()
        out.append(int((test_files == "Command and Control").sum()))
    return out
 
print("GroupKFold          :", c2_per_fold(GroupKFold(n_splits=5)))
print("StratifiedGroupKFold:", c2_per_fold(group_cv))

GroupKFold          : [0, 1, 1, 1, 2]
StratifiedGroupKFold: [1, 1, 1, 1, 1]


In [7]:
# Phép thử đường tắt: chỉ dùng tên máy

# Mô hình chỉ thấy cột Computer (mỗi tên máy là một token). Tính macro-F1 với chia ngẫu nhiên và chia theo nhóm.
# Sau đó thử thêm host_<tên máy> vào cuối tài liệu và chạy lại cấu hình A theo nhóm
host = df["Computer"].astype(str)
host_model = Pipeline([
    # cả tên máy là 1 token
    ("tfidf", TfidfVectorizer(token_pattern=r"[^\s]+")),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=3000))])
print("số máy khác nhau:", host.nunique())
# s_r = ____  : điểm CV (ngẫu nhiên) khi chỉ dùng tên máy
# s_g = ____   : điểm CV (theo nhóm) khi chỉ dùng tên máy
s_r = cross_validate(host_model, host, y, cv=random_cv, scoring="f1_macro")["test_score"]
s_g = cross_validate(host_model, host, y, cv=group_cv, groups=groups,
                     scoring="f1_macro")["test_score"]
print("chỉ tên máy - ngẫu nhiên:", round(s_r.mean(), 3))
print("chỉ tên máy - theo nhóm :", round(s_g.mean(), 3))
 
# THỬ thêm tên máy vào tài liệu
docs_host = docs + " host_" + host.str.lower()
s = cross_validate(make_model(), docs_host, y, cv=group_cv, groups=groups,
                   scoring="f1_macro")["test_score"]
print("docs + tên máy (theo nhóm):", round(s.mean(), 3))

số máy khác nhau: 19
chỉ tên máy - ngẫu nhiên: 0.349
chỉ tên máy - theo nhóm : 0.063
docs + tên máy (theo nhóm): 0.277


In [ ]:
# Import log_run, ghi 3 lần chạy: leak-random, leak-group, host-only-group.
# Đọc lại results.csv và in (bỏ cột MD5).
Path("../results/results.csv").unlink(missing_ok=True)
import src.baseline
importlib.reload(src.baseline)
from src.baseline import log_run

print(r)
 
log_run("leak-random", "StratifiedKFold(5,rs=42)", r["test_f1_macro"], C=1.0)
log_run("leak-group", "StratifiedGroupKFold(5,rs=42)", g["test_f1_macro"], C=1.0)
log_run("host-only-group", "StratifiedGroupKFold(5,rs=42)", s_g, C=1.0)
runs = pd.read_csv("../results/results.csv")
print(runs.drop(columns="data_md5").to_string(index=False))   # bỏ cột md5 cho gọn

         run_id                         split  macro_f1_mean  macro_f1_std sklearn   C
    leak-random      StratifiedKFold(5,rs=42)          0.803         0.009   1.9.1 1.0
     leak-group StratifiedGroupKFold(5,rs=42)          0.206         0.033   1.9.1 1.0
host-only-group StratifiedGroupKFold(5,rs=42)          0.063         0.038   1.9.1 1.0
